# **Proyecto Etapa 3 — Aprendizaje Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

**Actividad individual**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A01139580 |

---
## 1. Introducción: Aprendizaje Supervisado

### 1.1 Concepto general

El **aprendizaje supervisado** es una rama del aprendizaje automático en la que un modelo es entrenado a partir de un conjunto de pares entrada–salida etiquetados $(x_i, y_i)$. El objetivo es aprender una función $f : X \rightarrow Y$ que generalice bien a datos no vistos, minimizando alguna función de pérdida sobre el conjunto de entrenamiento.


Los dos grandes tipos de tareas supervisadas son:
- **Clasificación**: $Y$ es un conjunto discreto de clases (e.g., predecir el tipo de tejido).
- **Regresión**: $Y \subseteq \mathbb{R}$ (e.g., predecir un valor continuo de expresión génica).

---

### 1.2 Algoritmos representativos en la literatura

| Algoritmo | Tipo | Fortalezas | Limitaciones |
|-----------|------|------------|--------------|
| **Árbol de Decisión** (Decision Tree) | Clasificación / Regresión | Interpretable, no requiere normalización | Sobreajuste fácil; inestable ante pequeñas variaciones |
| **Random Forest** | Clasificación / Regresión | Robusto al sobreajuste; maneja alta dimensionalidad | Menos interpretable; costoso en memoria |
| **Gradient Boosted Trees (GBT)** | Clasificación / Regresión | Alta precisión; captura interacciones no lineales | Entrenamiento secuencial; más lento que RF |
| **Regresión Logística** | Clasificación | Simple, interpretable, probabilístico | Supone linealidad; sensible a multicolinealidad |
| **SVM (Support Vector Machine)** | Clasificación / Regresión | Efectivo en alta dimensión; margen máximo | Difícil de escalar a millones de instancias |
| **Perceptrón Multicapa (MLP)** | Clasificación / Regresión | Captura relaciones muy complejas | Requiere mucho dato y tunning cuidadoso |
| **Naive Bayes** | Clasificación | Muy rápido; bien calibrado con poco dato | Asume independencia entre features |

---

### 1.3 Algoritmos disponibles en PySpark MLlib

PySpark MLlib (`pyspark.ml.classification`) ofrece implementaciones distribuidas de los siguientes algoritmos de clasificación:

| Clase PySpark | Algoritmo |
|---------------|-----------|
| `DecisionTreeClassifier` | Árbol de decisión |
| `RandomForestClassifier` | Random Forest |
| `GBTClassifier` | Gradient Boosted Trees |
| `LogisticRegression` | Regresión logística (multinomial) |
| `MultilayerPerceptronClassifier` | Red neuronal MLP |
| `LinearSVC` | SVM lineal |
| `NaiveBayes` | Naive Bayes |
| `FMClassifier` | Factorization Machines |

El pipeline de MLlib sigue el patrón `Transformer → Estimator → Model`, compatible con `Pipeline` y `CrossValidator` para validación cruzada distribuida.

---

### 1.4 Algoritmo seleccionado: Random Forest

Se selecciona **Random Forest** por las siguientes razones aplicadas al contexto GTEx:

1. **Alta dimensionalidad**: cada muestra tiene miles de features (genes). RF selecciona aleatoriamente un subconjunto de features en cada split, lo que reduce la varianza sin requerir selección manual de genes.
2. **Robustez al sobreajuste**: el promedio de múltiples árboles decorrelacionados mitiga el sobreajuste que sufriría un árbol individual sobre datos de expresión génica ruidosos.
3. **Importancia de variables**: RF produce un ranking de importancia de genes (`featureImportances`), útil para interpretar qué genes distinguen los grupos de tejido.
4. **No requiere normalización estricta**: los valores TPM ya son comparables entre muestras, pero RF es insensible a escalas, a diferencia de MLP o SVM.

---
## 2. Selección de los datos

### 2.1 Estrategia

Se construye una sub-muestra **M'** a partir de muestras de tejido **cardiovascular** (corazón, vasos sanguíneos) y **musculoesquelético** (músculo, piel, tejido adiposo) del dataset GTEx V10:

- Muestreo estratificado de **200 muestras por grupo de tejido y sexo** (~800 muestras totales), suficiente para entrenar un clasificador multi-clase sin tiempos de procesamiento excesivos.
- Las features son los valores de expresión TPM de los 500 genes con mayor varianza entre las muestras de M'.

**Justificación del dominio:** GTEx representa la línea base de expresión génica normal en humanos en Tierra. La pregunta biológicamente relevante es si un modelo puede identificar el sub-tipo de tejido a partir de su perfil de expresión génica, lo cual podría detectar desvíos en muestras de astronautas.

In [1]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

TARGET_TISSUE_GROUPS  = ['Cardiovascular', 'Musculoesqueletico']
SAMPLES_PER_PARTITION = 200  
N_GENES_SUBSAMPLE     = 500   

print(f'Semilla aleatoria : {RANDOM_SEED}')
print(f'Grupos de tejido  : {TARGET_TISSUE_GROUPS}')
print(f'Muestras/partición: {SAMPLES_PER_PARTITION}')
print(f'Genes seleccionados: {N_GENES_SUBSAMPLE}')
print(f'JAVA_HOME seteado : {os.environ.get("JAVA_HOME", "ERROR - no seteado")}')

Semilla aleatoria : 42
Grupos de tejido  : ['Cardiovascular', 'Musculoesqueletico']
Muestras/partición: 200
Genes seleccionados: 500
JAVA_HOME seteado : C:\Users\diego\anaconda3\envs\big-data\Library\lib\jvm


In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_SupervisedLearning_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark

In [3]:

sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner')

tissue_group_col = F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso') \
    .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico') \
    .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular') \
    .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico') \
    .otherwise('Visceral_Metabolico')

sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = meta_df \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

# Filtrar a los grupos (P05, P06, P07, P08)
meta_target = meta_df.filter(F.col('TISSUE_GROUP').isin(TARGET_TISSUE_GROUPS))

print('Distribución por partición (target):')
meta_target.groupBy('TISSUE_GROUP', 'SEX_LABEL').count().orderBy('TISSUE_GROUP', 'SEX_LABEL').show()

Distribución por partición (target):


+------------------+---------+-----+
|      TISSUE_GROUP|SEX_LABEL|count|
+------------------+---------+-----+
|    Cardiovascular| Femenino|  771|
|    Cardiovascular|Masculino| 1573|
|Musculoesqueletico| Femenino| 1349|
|Musculoesqueletico|Masculino| 2827|
+------------------+---------+-----+



In [4]:
# --- Construir M ---
from collections import defaultdict

meta_rows = meta_target.select('COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD').collect()

partition_samples = defaultdict(list)
for row in meta_rows:
    key = (row['TISSUE_GROUP'], row['SEX_LABEL'])
    partition_samples[key].append((row['COL_NAME'], row['SMTSD']))

rng = random.Random(RANDOM_SEED)
selected_col_names = []
selected_meta = []   # [(col_name, tissue_group, sex_label)]

for (tg, sx), items in sorted(partition_samples.items()):
    n_partition = len(items)
    n_select = min(SAMPLES_PER_PARTITION, n_partition)

    by_subtype = defaultdict(list)
    for col, smtsd in items:
        by_subtype[smtsd].append(col)

    sampled = []
    for smtsd, cols in by_subtype.items():
        n_strata = max(1, round(n_select * len(cols) / n_partition))
        n_strata = min(n_strata, len(cols))
        sampled.extend(rng.sample(cols, n_strata))

    sampled = sampled[:n_select]

    for col in sampled:
        selected_col_names.append(col)
        selected_meta.append((col, tg, sx))

    print(f'  {tg:<22} + {sx:<12}: {len(sampled)} muestras seleccionadas de {n_partition}')

print(f'\nTotal M\' : {len(selected_col_names)} muestras')

  Cardiovascular         + Femenino    : 200 muestras seleccionadas de 771
  Cardiovascular         + Masculino   : 199 muestras seleccionadas de 1573
  Musculoesqueletico     + Femenino    : 200 muestras seleccionadas de 1349
  Musculoesqueletico     + Masculino   : 200 muestras seleccionadas de 2827

Total M' : 799 muestras


In [5]:
import pandas as pd
from pyspark.sql.functions import split as spark_split

peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_col_names_raw = peek.columns.tolist()
all_col_names_clean = [c.replace('-', '_').replace('.', '_') for c in all_col_names_raw]

name_to_idx = {clean: idx for idx, clean in enumerate(all_col_names_clean)}

fixed_cols = ['Name', 'Description']
fixed_indices = [name_to_idx[c] for c in fixed_cols]

valid_sample_cols = [c for c in selected_col_names if c in name_to_idx]
sample_indices = [name_to_idx[c] for c in valid_sample_cols]

all_selected_indices = fixed_indices + sample_indices
all_selected_names   = fixed_cols + valid_sample_cols

print(f'Columnas de muestra válidas: {len(valid_sample_cols)} / {len(selected_col_names)}')

raw_df = spark.read.text(FILE_PATH)
gene_df = raw_df.filter(F.col('value').startswith('ENSG'))
split_col = spark_split(F.col('value'), '\t')

df_tpm = gene_df.select(
    *[split_col.getItem(i).alias(all_selected_names[idx])
      for idx, i in enumerate(all_selected_indices)]
)

for c in valid_sample_cols:
    df_tpm = df_tpm.withColumn(c, F.col(c).cast(FloatType()))

print(f'DataFrame TPM M\': {df_tpm.count():,} genes x {len(valid_sample_cols)} muestras')
df_tpm.select(all_selected_names[:5]).show(3)

Columnas de muestra válidas: 799 / 799


DataFrame TPM M': 59,033 genes x 799 muestras


+-----------------+-----------+-----------------------+------------------------+------------------------+
|             Name|Description|GTEX_QDT8_0426_SM_32PKZ|GTEX_13CF3_2226_SM_5J2MX|GTEX_11GSP_2926_SM_5N9C2|
+-----------------+-----------+-----------------------+------------------------+------------------------+
|ENSG00000223972.5|    DDX11L1|                    0.0|                     0.0|                     0.0|
|ENSG00000227232.5|     WASH7P|                5.78407|                 3.13248|                 3.17456|
|ENSG00000278267.1|  MIR6859-1|                    0.0|                     0.0|                     0.0|
+-----------------+-----------+-----------------------+------------------------+------------------------+
only showing top 3 rows


---
## 3. Selección de Features por Varianza

La selección aleatoria de genes no garantiza incluir los más informativos para la tarea. Se implementa **selección por varianza**: tomar los 500 genes con mayor varianza entre las 799 muestras de M'.

**Justificación biológica:** Los genes con alta varianza entre muestras de distintos tejidos son candidatos naturales a marcadores tisulares. Genes como troponinas cardíacas, actinas de músculo liso y miosinas tienen expresión altísima en tejido cardiovascular y baja en musculoesquelético (o viceversa), produciendo alta varianza inter-muestra. La selección por varianza captura estas diferencias sin necesidad de conocimiento previo.

In [6]:

print("Cargando matriz completa para cálculo de varianza...")
tpm_full_pd = df_tpm.select(['Name'] + valid_sample_cols).toPandas()
tpm_full_pd = tpm_full_pd.set_index('Name')

gene_var = tpm_full_pd.var(axis=1).sort_values(ascending=False)
print(f"Genes con varianza calculada: {len(gene_var):,}")
print("\nTop 10 genes por varianza:")
print(gene_var.head(10))

Cargando matriz completa para cálculo de varianza...


Genes con varianza calculada: 59,033

Top 10 genes por varianza:
Name
ENSG00000198804.2     566183744.0
ENSG00000198938.2     430945984.0
ENSG00000198899.2     406831616.0
ENSG00000198886.2     394051104.0
ENSG00000198712.1     296982080.0
ENSG00000210082.2     195647664.0
ENSG00000198888.2     173899232.0
ENSG00000198727.2     166914752.0
ENSG00000175206.11    163619712.0
ENSG00000198763.3     139426624.0
dtype: float32


In [7]:
TOP_N = 500
top_var_ids = gene_var.head(TOP_N).index.tolist()

print(f"Top {TOP_N} genes seleccionados por varianza.")
print(f"\nTop 10 genes con mayor varianza (candidatos a marcadores tisulares):")
desc_map = df_tpm.select('Name', 'Description').toPandas().set_index('Name')['Description'].to_dict()
for eid in top_var_ids[:10]:
    sym = desc_map.get(eid, '?')
    print(f"  {eid:<30} {sym:<20}  varianza={gene_var[eid]:.0f}")

Top 500 genes seleccionados por varianza.

Top 10 genes con mayor varianza (candidatos a marcadores tisulares):


  ENSG00000198804.2              MT-CO1                varianza=566183744
  ENSG00000198938.2              MT-CO3                varianza=430945984
  ENSG00000198899.2              MT-ATP6               varianza=406831616
  ENSG00000198886.2              MT-ND4                varianza=394051104
  ENSG00000198712.1              MT-CO2                varianza=296982080
  ENSG00000210082.2              MT-RNR2               varianza=195647664
  ENSG00000198888.2              MT-ND1                varianza=173899232
  ENSG00000198727.2              MT-CYB                varianza=166914752
  ENSG00000175206.11             NPPA                  varianza=163619712
  ENSG00000198763.3              MT-ND2                varianza=139426624


In [8]:
# --- Build feature matrix with variance-selected genes ---
rename_map_v2 = {g: g.replace('.', '_') for g in top_var_ids}
opt_genes_in_matrix = [g for g in top_var_ids if g in tpm_full_pd.index]

tpm_opt = tpm_full_pd.loc[opt_genes_in_matrix].T.reset_index()
tpm_opt = tpm_opt.rename(columns={'index': 'COL_NAME'})
tpm_opt = tpm_opt.rename(columns={g: g.replace('.', '_') for g in opt_genes_in_matrix})
gene_cols_v2 = [g.replace('.', '_') for g in opt_genes_in_matrix]
tpm_opt[gene_cols_v2] = tpm_opt[gene_cols_v2].fillna(0.0)

print(f"Matriz de features: {tpm_opt.shape[0]} muestras x {len(gene_cols_v2)} genes")

Matriz de features: 799 muestras x 500 genes


---
## 4. Experimento: Clasificación Multi-clase con Split por Donante

### 4.1 Variable objetivo y tarea

**Variable objetivo:** `SMTSD` (sub-tipo de tejido) — clasificación multi-clase con **11 clases**: tres tipos de arteria, dos de corazón, dos de piel, dos de tejido adiposo, músculo esquelético y fibroblastos.

**Justificación:** Predecir el sub-tipo de tejido a partir del perfil de expresión génica es un problema genuinamente difícil — los sub-tipos dentro del mismo grupo comparten la mayoría de genes expresados y solo difieren en marcadores específicos (e.g., PLN y ACTN2 distinguen ventrículo izquierdo de aurícula; ambos expresan el repertorio completo de genes cardíacos). Esta granularidad es relevante para el objetivo del proyecto: un modelo que aprende firmas de expresión por sub-tipo podría detectar alteraciones específicas de tejido en muestras de astronautas.

### 4.2 División por donante

Un mismo donante puede aportar muestras de varios sub-tipos de tejido. Si la división se hace por muestra, muestras del mismo donante aparecen en train y test simultáneamente, permitiendo que el modelo memorice perfiles individuales en lugar de aprender patrones generalizables. Se utiliza `GroupShuffleSplit` por `SUBJID` con proporción 80/20 — ningún donante aparece en ambos conjuntos, lo que fuerza al modelo a generalizar a individuos nuevos.

### 4.3 Pipeline MLlib

1. **`VectorAssembler`**: convierte las columnas de genes (top-500 por varianza) en un vector de features denso.
2. **`RandomForestClassifier`**: 100 árboles, profundidad máxima 10, `featureSubsetStrategy='sqrt'` (≈22 genes por split).
3. **Evaluación**: Accuracy y F1 macro como métricas principales, complementadas con reporte por clase.

In [9]:
meta_pd = meta_target.select('COL_NAME', 'SMTSD', 'SUBJID').toPandas()
meta_pd = meta_pd.drop_duplicates(subset='COL_NAME')

tpm_exp3 = tpm_opt[['COL_NAME'] + gene_cols_v2].merge(meta_pd, on='COL_NAME', how='inner')
tpm_exp3 = tpm_exp3.dropna(subset=['SMTSD', 'SUBJID'])

print(f'Matriz Experimento 3: {tpm_exp3.shape[0]} muestras x {len(gene_cols_v2)} features')
print('\nDistribución por SMTSD (sub-tipo de tejido):')
print(tpm_exp3['SMTSD'].value_counts())

Matriz Experimento 3: 799 muestras x 500 features

Distribución por SMTSD (sub-tipo de tejido):
SMTSD
Artery - Tibial                        117
Artery - Aorta                          82
Muscle - Skeletal                       78
Heart - Atrial Appendage                77
Heart - Left Ventricle                  76
Skin - Sun Exposed (Lower leg)          72
Adipose - Subcutaneous                  69
Cells - Cultured fibroblasts            63
Skin - Not Sun Exposed (Suprapubic)     62
Adipose - Visceral (Omentum)            56
Artery - Coronary                       47
Name: count, dtype: int64


In [10]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

le_smtsd = LabelEncoder()
tpm_exp3 = tpm_exp3.copy()
tpm_exp3['smtsd_label'] = le_smtsd.fit_transform(tpm_exp3['SMTSD'])
n_classes = len(le_smtsd.classes_)

print(f'Sub-tipos de tejido ({n_classes} clases):')
for code, name in enumerate(le_smtsd.classes_):
    count = (tpm_exp3['smtsd_label'] == code).sum()
    print(f'  {code}: {name} ({count} muestras)')

X_exp3  = tpm_exp3[gene_cols_v2].values
y_exp3  = tpm_exp3['smtsd_label'].values
groups  = tpm_exp3['SUBJID'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X_exp3, y_exp3, groups=groups))

X_train_e3, X_test_e3 = X_exp3[train_idx], X_exp3[test_idx]
y_train_e3, y_test_e3 = y_exp3[train_idx], y_exp3[test_idx]

train_donors = set(groups[train_idx])
test_donors  = set(groups[test_idx])
overlap = train_donors & test_donors
print(f'\nTrain: {len(X_train_e3)} muestras, {len(train_donors)} donantes únicos')
print(f'Test : {len(X_test_e3)} muestras, {len(test_donors)} donantes únicos')
print(f'Donantes en ambos conjuntos: {len(overlap)} (debe ser 0)')

Sub-tipos de tejido (11 clases):
  0: Adipose - Subcutaneous (69 muestras)
  1: Adipose - Visceral (Omentum) (56 muestras)
  2: Artery - Aorta (82 muestras)
  3: Artery - Coronary (47 muestras)
  4: Artery - Tibial (117 muestras)
  5: Cells - Cultured fibroblasts (63 muestras)
  6: Heart - Atrial Appendage (77 muestras)
  7: Heart - Left Ventricle (76 muestras)
  8: Muscle - Skeletal (78 muestras)
  9: Skin - Not Sun Exposed (Suprapubic) (62 muestras)
  10: Skin - Sun Exposed (Lower leg) (72 muestras)

Train: 644 muestras, 412 donantes únicos
Test : 155 muestras, 104 donantes únicos
Donantes en ambos conjuntos: 0 (debe ser 0)


In [11]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

train_pd_e3 = pd.DataFrame(X_train_e3, columns=gene_cols_v2)
train_pd_e3['label'] = y_train_e3.tolist()
test_pd_e3  = pd.DataFrame(X_test_e3, columns=gene_cols_v2)
test_pd_e3['label']  = y_test_e3.tolist()

train_spark_e3 = spark.createDataFrame(train_pd_e3)
test_spark_e3  = spark.createDataFrame(test_pd_e3)

assembler_e3 = VectorAssembler(inputCols=gene_cols_v2, outputCol='features')
rf_e3 = RandomForestClassifier(
    labelCol='label', featuresCol='features',
    numTrees=100, maxDepth=10, featureSubsetStrategy='sqrt', seed=RANDOM_SEED
)
pipeline_e3 = Pipeline(stages=[assembler_e3, rf_e3])

print('Entrenando Random Forest (multi-clase SMTSD + split por donante)...')
model_e3 = pipeline_e3.fit(train_spark_e3)
print('Entrenamiento completado.')

Entrenando Random Forest (multi-clase SMTSD + split por donante)...


Entrenamiento completado.


In [12]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.metrics import classification_report

preds_e3 = model_e3.transform(test_spark_e3)

acc_e3_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy')
f1_e3_eval = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1')

acc_e3 = acc_e3_eval.evaluate(preds_e3)
f1_e3  = f1_e3_eval.evaluate(preds_e3)

preds_e3_pd = preds_e3.select('label', 'prediction').toPandas()
print(f'Accuracy (macro): {acc_e3:.4f}')
print(f'F1-Score (macro): {f1_e3:.4f}')
print()
print('Reporte por sub-tipo de tejido:')
print(classification_report(
    preds_e3_pd['label'].astype(int),
    preds_e3_pd['prediction'].astype(int),
    target_names=le_smtsd.classes_
))

Accuracy (macro): 0.9613
F1-Score (macro): 0.9609

Reporte por sub-tipo de tejido:
                                     precision    recall  f1-score   support

             Adipose - Subcutaneous       1.00      1.00      1.00        13
       Adipose - Visceral (Omentum)       1.00      1.00      1.00        10
                     Artery - Aorta       0.94      1.00      0.97        15
                  Artery - Coronary       1.00      0.83      0.91        12
                    Artery - Tibial       0.96      1.00      0.98        26
       Cells - Cultured fibroblasts       1.00      1.00      1.00        12
           Heart - Atrial Appendage       1.00      1.00      1.00        17
             Heart - Left Ventricle       1.00      1.00      1.00        14
                  Muscle - Skeletal       1.00      1.00      1.00        11
Skin - Not Sun Exposed (Suprapubic)       0.86      0.86      0.86        14
     Skin - Sun Exposed (Lower leg)       0.82      0.82      0.82   

In [13]:
# --- Resultados finales ---
print('=' * 50)
print('  RESULTADOS DEL MODELO')
print('=' * 50)
print(f'  Accuracy         : {acc_e3:.4f}')
print(f'  F1-Score (macro) : {f1_e3:.4f}')
print('=' * 50)
print()
print(f'  Tarea  : clasificación multi-clase SMTSD ({n_classes} sub-tipos)')
print('  Split  : GroupShuffleSplit por donante (0 donantes en común)')
print('  Semilla: RANDOM_SEED = 42')

  RESULTADOS DEL MODELO
  Accuracy         : 0.9613
  F1-Score (macro) : 0.9609

  Tarea  : clasificación multi-clase SMTSD (11 sub-tipos)
  Split  : GroupShuffleSplit por donante (0 donantes en común)
  Semilla: RANDOM_SEED = 42


---
### 4.4 Interpretación de resultados

#### Métricas obtenidas

| Métrica | Valor |
|---------|-------|
| **Accuracy** | **96.1%** |
| **F1-Score (macro)** | **0.961** |

#### Análisis por sub-tipo de tejido

Los sub-tipos más difíciles de clasificar son los dos tipos de piel (Skin Sun Exposed: 82% F1; Skin Not Sun Exposed: 86% F1), que comparten la mayoría del transcriptoma y se diferencian principalmente por genes de respuesta a radiación UV. Los tipos cardíacos, musculares y adiposos alcanzan 100% de precisión, lo que refleja firmas génicas claramente diferenciadas: PLN y ACTN2 para corazón; MYL9 y MYOZ2 para músculo esquelético.

#### Impacto del split por donante

Al garantizar que ningún donante aparece simultáneamente en train y test, el modelo debe generalizar a individuos nuevos. En RNA-seq, la variabilidad inter-individual puede ser significativa (factores genéticos, ambiente, edad). Este split refleja la capacidad predictiva real sobre nuevas personas — requisito esencial para aplicaciones en medicina espacial donde se compararían perfiles de astronautas individuales contra esta línea base.

#### Relevancia para medicina espacial

Un modelo que discrimina 11 sub-tipos de tejido controlando la variabilidad individual proporciona una línea base robusta para comparar con datos de astronautas. Los cambios en expresión génica inducidos por microgravedad se detectarían sobre esta línea base sin confundir señal biológica real con artefactos de variabilidad individual.

---
## Referencias

1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
2. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
3. Apache Spark MLlib. (2024). Classification and regression. https://spark.apache.org/docs/latest/ml-classification-regression.html
4. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.

---

## Declaración de uso de Inteligencia Artificial

Anthropic. (2026). *Claude Sonnet 4.6* [Modelo de lenguaje grande], utilizado para soporte en estructura del notebook, documentación de celdas markdown y revisión del código PySpark. https://claude.ai

*La responsabilidad final sobre el contenido entregado recae en el autor. Las decisiones de diseño del experimento, selección de algoritmo, variable objetivo y la interpretación de resultados son del autor.*